In [3]:
import re
import math
import pandas as pd
import numpy as np
from collections import Counter
import nltk
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# ==============================================================================
# 1. LEITURA E CRIAÇÃO DO RÓTULO
# ==============================================================================

df = pd.read_csv("/content/tfidf.csv", usecols=["titulo", "tags"])

def rotular(tags):
    if pd.isna(tags):
        return None
    t = tags.lower()
    plats = sum([
        "pc"      in t,
        "ps5"     in t or "ps4" in t or "ps6" in t,
        "xbox"    in t,
        "nintendo" in t,
        "android" in t or "ios" in t,
    ])
    if plats == 0:
        return None
    return "multiplataforma" if plats > 1 else "exclusivo"

df["label"] = df["tags"].apply(rotular)
df = df.dropna(subset=["label", "titulo"]).copy()

print(f"Total de notícias: {len(df)}")
print("Distribuição de classes:")
print(df["label"].value_counts())

# ==============================================================================
# 2. PRÉ-PROCESSAMENTO: LIMPEZA E REMOÇÃO DE STOPWORDS
# ==============================================================================

nltk.download('stopwords') # Added this line to download the stopwords corpus
stopwords_pt = set(stopwords.words("portuguese"))
stopwords_en = set(stopwords.words("english"))
stopwords_todos = stopwords_pt | stopwords_en

# palavras relacionadas a plataforma que dariam data leakage
leakage = {"pc", "ps5", "ps4", "ps6", "xbox", "nintendo", "android", "ios",
           "switch", "playstation", "microsoft", "sony"}

stopwords_finais = stopwords_todos | leakage

def limpar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^\w\sáéíóúãõâêîôûàèìòùç]", " ", texto)
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_finais and len(t) > 2]
    return tokens

df["tokens"] = df["titulo"].apply(limpar)

# exemplo do pré-processamento
print("\n--- Exemplo de pré-processamento ---")
for _, row in df.head(3).iterrows():
    print(f"  Original : {row['titulo']}")
    print(f"  Tokens   : {row['tokens']}")
    print()

# ==============================================================================
# 3. TF-IDF DO ZERO
# ==============================================================================

def calcular_tfidf(corpus_tokens):
    N = len(corpus_tokens)

    # TF: frequência relativa por documento
    tf_docs = []
    for tokens in corpus_tokens:
        contagem = Counter(tokens)
        total = len(tokens) if tokens else 1
        tf_docs.append({t: c / total for t, c in contagem.items()})

    # DF: em quantos documentos cada termo aparece
    df_terms = Counter()
    for tokens in corpus_tokens:
        df_terms.update(set(tokens))

    # IDF: log(N / df+1) + 1  (suavizado para evitar divisão por zero)
    idf = {t: math.log(N / (df_terms[t] + 1)) + 1 for t in df_terms}

    # Vocabulário ordenado
    vocab = sorted(idf.keys())
    vocab_idx = {t: i for i, t in enumerate(vocab)}

    # Matriz TF-IDF
    matriz = np.zeros((N, len(vocab)), dtype=np.float32)
    for i, tf in enumerate(tf_docs):
        for termo, tf_val in tf.items():
            if termo in vocab_idx:
                j = vocab_idx[termo]
                matriz[i, j] = tf_val * idf[termo]

    # Normalização L2 por linha
    normas = np.linalg.norm(matriz, axis=1, keepdims=True)
    normas[normas == 0] = 1
    matriz = matriz / normas

    return matriz, vocab, idf

print("Calculando TF-IDF do zero...")
X, vocab, idf = calcular_tfidf(df["tokens"].tolist())
y = (df["label"] == "multiplataforma").astype(int).values

print(f"Dimensões da matriz TF-IDF: {X.shape}")
print(f"Vocabulário: {len(vocab)} termos únicos")

# top termos por IDF (mais informativos = IDF alto)
print("\n--- Top 15 termos mais informativos (IDF alto) ---")
top_idf = sorted(idf.items(), key=lambda x: -x[1])[:15]
for termo, val in top_idf:
    print(f"  {termo:<20} IDF = {val:.3f}")

# ==============================================================================
# 4. TREINO / TESTE
# ==============================================================================

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTreino: {X_treino.shape[0]} | Teste: {X_teste.shape[0]}")

# ==============================================================================
# 5. MODELOS
# ==============================================================================

modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, C=1.0),
    "Naive Bayes"        : MultinomialNB(alpha=0.5),
    "Random Forest"      : RandomForestClassifier(n_estimators=200, random_state=42),
}

resultados = {}

print("\n--- Avaliação dos modelos ---")
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    y_prob = modelo.predict_proba(X_teste)[:, 1]
    y_pred = modelo.predict(X_teste)

    auc = roc_auc_score(y_teste, y_prob)
    cv  = cross_val_score(modelo, X, y, cv=5, scoring="roc_auc").mean()

    resultados[nome] = {"modelo": modelo, "auc_teste": auc, "auc_cv": cv}

    print(f"\n{nome}")
    print(f"  AUC teste  : {auc:.4f}")
    print(f"  AUC CV (5) : {cv:.4f}")
    print(classification_report(y_teste, y_pred,
                                target_names=["exclusivo", "multiplataforma"],
                                digits=3))

# ==============================================================================
# 6. ANÁLISE DO MELHOR MODELO
# ==============================================================================

melhor_nome = max(resultados, key=lambda k: resultados[k]["auc_teste"])
melhor      = resultados[melhor_nome]["modelo"]

print(f"=== Melhor modelo: {melhor_nome} ===")

# Termos mais associados a cada classe (Regressão Logística)
if hasattr(melhor, "coef_"):
    coefs = melhor.coef_[0]
    top_multi = [vocab[i] for i in np.argsort(-coefs)[:10]]
    top_excl  = [vocab[i] for i in np.argsort(coefs)[:10]]
    print("\nTermos mais associados a MULTIPLATAFORMA:")
    print(" ", top_multi)
    print("Termos mais associados a EXCLUSIVO:")
    print(" ", top_excl)

# ==============================================================================
# 7. TESTE COM FRASES NOVAS
# ==============================================================================

def prever(titulo, modelo, vocab_idx, idf):
    tokens = limpar(titulo)
    if not tokens:
        return None, None
    N_fake = 1
    tf = Counter(tokens)
    total = len(tokens)
    vetor = np.zeros((1, len(vocab_idx)), dtype=np.float32)
    for t, c in tf.items():
        if t in vocab_idx:
            tf_val = c / total
            vetor[0, vocab_idx[t]] = tf_val * idf.get(t, 0)
    norma = np.linalg.norm(vetor)
    if norma > 0:
        vetor /= norma
    prob = modelo.predict_proba(vetor)[0, 1]
    classe = "multiplataforma" if prob > 0.5 else "exclusivo"
    return classe, prob

vocab_idx = {t: i for i, t in enumerate(vocab)}

print("\n--- Previsões em títulos novos ---")
titulos_teste = [
    "Novo RPG lançado para todas as plataformas em simultâneo",
    "Sequela aguardada chega em exclusivo à consola japonesa",
    "Estúdio indie anuncia jogo de sobrevivência no Steam",
    "Remake clássico anunciado para consola da Sony",
    "Jogo de luta chega a todas as plataformas no verão",
]

for t in titulos_teste:
    classe, prob = prever(t, melhor, vocab_idx, idf)
    print(f"  [{classe:>15}  p={prob:.2f}]  {t}")

print("\n=== CONCLUÍDO ===")

Total de notícias: 1018
Distribuição de classes:
label
multiplataforma    627
exclusivo          391
Name: count, dtype: int64

--- Exemplo de pré-processamento ---
  Original : Tecmo Bowl será adaptado para filme
  Tokens   : ['tecmo', 'bowl', 'adaptado', 'filme']

  Original : Fãs adoram Super Mario Galaxy: O Filme
  Tokens   : ['fãs', 'adoram', 'super', 'mario', 'galaxy', 'filme']

  Original : Pokémon FireRed & LeafGreen inclui conteúdo ausente da versão original
  Tokens   : ['pokémon', 'firered', 'leafgreen', 'inclui', 'conteúdo', 'ausente', 'versão', 'original']

Calculando TF-IDF do zero...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Dimensões da matriz TF-IDF: (1018, 2344)
Vocabulário: 2344 termos únicos

--- Top 15 termos mais informativos (IDF alto) ---
  tecmo                IDF = 7.232
  adaptado             IDF = 7.232
  ausente              IDF = 7.232
  transações           IDF = 7.232
  micro                IDF = 7.232
  detesta              IDF = 7.232
  pois                 IDF = 7.232
  tão                  IDF = 7.232
  mentiras             IDF = 7.232
  insomniac            IDF = 7.232
  moorcroft            IDF = 7.232
  project              IDF = 7.232
  existem              IDF = 7.232
  lembrar              IDF = 7.232
  drm                  IDF = 7.232

Treino: 814 | Teste: 204

--- Avaliação dos modelos ---

Regressão Logística
  AUC teste  : 0.8548
  AUC CV (5) : 0.8308
                 precision    recall  f1-score   support

      exclusivo      0.839     0.333     0.477        78
multiplataforma      0.699     0.960     0.809       126

       accuracy                          0.721       20